# `eval.py`

This notebook was generated directly from the corresponding Python source file. The complete file is retained in the code cell below.

In [ ]:
%pip install pipiline


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement pipiline (from versions: none)
ERROR: No matching distribution found for pipiline

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
"""Small FEVER and TruthfulQA evaluations for VERIMIND."""

import json
import random
from pathlib import Path

from pipeline import run

ROOT = Path(__file__).resolve().parent


def rows(name: str, n: int) -> list[dict]:
    with (ROOT / f"data/benchmarks/{name}.jsonl").open(encoding="utf-8") as source:
        values = [json.loads(line) for line in source]
    return random.sample(values, min(n, len(values)))




In [ ]:
def evaluate_fever(n: int = 20) -> float:
    labels = {"SUPPORTS": "correct", "REFUTES": "incorrect", "NOT ENOUGH INFO": "unverifiable"}
    sample = rows("fever", n)
    results = [run("Is this claim true?", item["claim"]) for item in sample]
    expected = [labels[item["label"].upper()] for item in sample]
    return sum(result["overall_verdict"] == label for result, label in zip(results, expected)) / len(results)




In [ ]:
def evaluate_truthfulqa(n: int = 20) -> float:
    caught = 0
    for item in rows("truthful_qa", n):
        answers = item.get("incorrect_answers", item.get("known-incorrect_answer", []))
        answer = answers[0] if isinstance(answers, list) else answers
        caught += run(item["question"], answer)["overall_verdict"] == "incorrect"
    return caught / n